In [1466]:
board = [
[0,0,0],
[0,0,0],
[0,0,0]
]



In [1467]:
board

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]

In [1468]:
board[1][1] = 1

In [1469]:
board

[[0, 0, 0], [0, 1, 0], [0, 0, 0]]

In [1470]:
empty = 0
X = 1 
O = -1

In [1471]:
board[0][0] =X

In [1472]:
board[1][1] = O

In [1473]:
board

[[1, 0, 0], [0, -1, 0], [0, 0, 0]]

In [1474]:
def invalid(a,b):
    if board[a][b] == X or board[a][b] == O:
        return True

    return False

In [1475]:
def move(current_player,a,b):
  if current_player == X:
    board[a][b] = 1
  else:
     board[a][b] = -1


In [1476]:
def clear(board):
  for i in range (3):
    for j in range (3):
      board[i][j] = 0

In [1477]:
clear(board)

In [1478]:
board  

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]

In [1479]:
current_player = X
move(X,0,0)

In [1480]:
board

[[1, 0, 0], [0, 0, 0], [0, 0, 0]]

In [1481]:
board

[[1, 0, 0], [0, 0, 0], [0, 0, 0]]

In [1482]:
def check_win(board):
  if board[0][0] == board[0][1] == board [0][2] == 1 or board[1][0] == board[1][1] == board [1][2] == 1 or board[2][0] == board[2][1] == board [2][2] == 1:
    return X
  elif board[0][0] == board[1][0] == board [2][0] == 1 or board[0][1] == board[1][1] == board [2][1] == 1 or board[0][2] == board[1][2] == board [2][2] == 1:
    return X
  elif board[0][0] == board[1][1] == board[2][2] == 1:
    return X
  elif board[0][2] == board[1][1] == board[2][0] == 1:
    return X
  elif board[0][0] == board[0][1] == board [0][2] == -1 or board[1][0] == board[1][1] == board [1][2] == -1 or board[2][0] == board[2][1] == board [2][2] == -1:
    return O
  elif board[0][0] == board[1][0] == board [2][0] == -1 or board[0][1] == board[1][1] == board [2][1] == -1 or board[0][2] == board[1][2] == board [2][2] == -1:
    return O
  elif board[0][0] == board[1][1] == board[2][2] == -1:
    return O
  elif board[0][2] == board[1][1] == board[2][0] == -1:
    return O
  else:
    return None

In [1483]:
Draw = "Draw"
def check_draw(board):
  board_full = True
  for i in range (3):
    for j in range (3):
      if board[i][j] == 0:
        board_full= False
 
  if check_win(board) == None and board_full:
    return Draw
  

In [1484]:
import random
def random_move():

  row = random.randint(0,2)
  column = random.randint(0,2)
  return row,column

In [1485]:
clear(board)
print(board)


[[0, 0, 0], [0, 0, 0], [0, 0, 0]]


In [1486]:
def flatten(board):
  new_board = []
  for i in range (3):
    for j in range (3):
      new_board.append(board[i][j])
  return new_board 


In [1487]:
def indexing(row,column):
  index = row*3 + column
  return index


In [1488]:
X_train = []
y_train = []


In [1489]:
for game in range (5000):
  clear(board)
  history = []
  current_player = X
  while (check_win(board)!= X and check_win(board)!= O) and check_draw(board)!=Draw:
    board_snapshot = [row[:] for row in board]
    row , column = random_move()

    while invalid(row,column):
      row , column = random_move()
    if current_player == X:
      history.append([board_snapshot,
                (row,column),
                X])
      move(X, row, column)
      
      current_player = O

    else:
      history.append([board_snapshot,
                (row,column),
                O])
      move(O,row,column)
      
      current_player = X

  if check_win(board) == X:
    reward = 1

  elif check_win(board) == O:
    reward = -1

  elif check_draw(board) == Draw:
    reward = 0
  for i in history:
    i.append(reward)
  winner_moves = []
  loser_moves = []
  for i in range (len(history)):
    if (reward==1 and history[i][2] == 1):
      winner_moves.append(history[i])
    elif (reward==-1 and history[i][2] == -1):
       winner_moves.append(history[i])
      
    if (reward == 1) and history[i][2] == -1:
      loser_moves.append(history[i])
    elif reward == -1 and history[i][2] == 1:
      loser_moves.append(history[i])


  if len(loser_moves) > 0:

    bad_move = loser_moves[-1][1]


  if len(winner_moves) > 0:
    last_moves = winner_moves[-3:]

    for moves in last_moves:

      if len(loser_moves) > 0 and moves[1] == bad_move:
        continue

      X_train.append(flatten(moves[0]))
      y_train.append(indexing(*moves[1]))  

 


  



In [1490]:
#y_train

In [1491]:
print(len(X_train))
print(len(y_train))

13080
13080


In [1492]:
import torch

In [1493]:
X_train = torch.tensor(X_train)
y_train = torch.tensor(y_train)

In [1494]:
print(X_train.shape)
print(y_train.shape)

torch.Size([13080, 9])
torch.Size([13080])


# Defining the model

In [1495]:
# define NN class
import torch.nn as nn
class MyNN(nn.Module):
  def __init__(self, num_features):

    super().__init__()
    self.model = nn.Sequential(
      nn.Linear(num_features, 32),
      nn.ReLU(),
      nn.Linear(32,32),
      nn.ReLU(),
      nn.Linear(32, 9)
    )

  def forward(self, x):
      return self.model(x)

  

In [1496]:
model = MyNN(9)

In [1497]:
criterion = nn.CrossEntropyLoss()

In [1498]:
import torch.optim as optim
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [1499]:
X_train = X_train.float()
epochs = 100

# training loop

In [1500]:
for epoch in range(epochs):
  y_pred = model(X_train)
  loss = criterion(y_pred, y_train)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')


Epoch: 1, Loss: 2.222560405731201
Epoch: 2, Loss: 2.2199714183807373
Epoch: 3, Loss: 2.21746563911438
Epoch: 4, Loss: 2.2150371074676514
Epoch: 5, Loss: 2.2126893997192383
Epoch: 6, Loss: 2.2104201316833496
Epoch: 7, Loss: 2.2082340717315674
Epoch: 8, Loss: 2.2061214447021484
Epoch: 9, Loss: 2.2040750980377197
Epoch: 10, Loss: 2.2021000385284424
Epoch: 11, Loss: 2.2001843452453613
Epoch: 12, Loss: 2.1983251571655273
Epoch: 13, Loss: 2.1965315341949463
Epoch: 14, Loss: 2.1947989463806152
Epoch: 15, Loss: 2.193122148513794
Epoch: 16, Loss: 2.191495418548584
Epoch: 17, Loss: 2.1899285316467285
Epoch: 18, Loss: 2.188416004180908
Epoch: 19, Loss: 2.1869542598724365
Epoch: 20, Loss: 2.1855435371398926
Epoch: 21, Loss: 2.184178352355957
Epoch: 22, Loss: 2.182856798171997
Epoch: 23, Loss: 2.181581497192383
Epoch: 24, Loss: 2.180352210998535
Epoch: 25, Loss: 2.179159641265869
Epoch: 26, Loss: 2.1780037879943848
Epoch: 27, Loss: 2.1768832206726074
Epoch: 28, Loss: 2.1758031845092773
Epoch: 29, L

In [1501]:
sample = X_train[0]

In [1502]:
sample = sample.unsqueeze(0)

In [1503]:
output = model(sample)
output

tensor([[ 0.1012, -0.1383, -0.0581, -0.2280,  0.3471, -0.2115,  0.0836, -0.4256,
          0.1600]], grad_fn=<AddmmBackward0>)

In [1504]:
prediction = torch.argmax(output)
print(prediction)

tensor(4)


In [1505]:
clear(board)
print(board)

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]


In [1508]:

ai_wins = 0
o_wins = 0
draws = 0



for game in range(100):
  if game % 2 == 0:
    current_player = X
  else:
    current_player = O
  clear(board)
  while (check_win(board)!=X and check_win(board)!=O) and check_draw(board)!=Draw:
    if current_player == X:
      input = torch.tensor(flatten(board))
      input = input.float()
      input = input.unsqueeze(0)
      output = model(input)

      ranked_moves = torch.argsort(output, descending=True)

      for index in ranked_moves[0]:
        index = index.item()
        row = index // 3
        column = index % 3
        if not invalid(row,column):
         move(X,row,column)
         current_player = O
         break
    
    else:
      temp_board = []
      for row in board:
        temp_row = []

        for cell in row:
          temp_row.append(-cell)
        temp_board.append(temp_row)
    
      input = torch.tensor(flatten(temp_board))
      input = input.float()
      input = input.unsqueeze(0)
      output = model(input)

      ranked_moves = torch.argsort(output, descending=True)

      for index in ranked_moves[0]:
        index = index.item()
        row = index // 3
        column = index % 3
        if not invalid(row,column):
         move(O,row,column)
         current_player = X
         break
    
    
  if check_win(board) == X:
    ai_wins += 1

  elif check_win(board) == O:
    o_wins += 1

  else:
    draws += 1

In [1509]:

print(ai_wins)
print(o_wins)
print(draws)

50
50
0
